# Itty-Bitty-Piano Ablation Training (Kaggle)

This notebook trains one ablation variant (`A`, `B`, or `C`) on pre-tokenized Godzilla piano data,
then generates and visualizes one continuation MIDI.

Run top-to-bottom. Between variant runs, edit only the **CONFIG** cell.


## 1. ENVIRONMENT SETUP


In [ ]:
import time

TIMER_ENV_START = time.time()
print("ENVIRONMENT SETUP started...")


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Core package installs.
_ = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ncps",
        "pretty_midi",
        "safetensors",
        "miditok",
        "matplotlib",
        "huggingface_hub",
    ],
    check=False,
)

# Try real Mamba install, fail gracefully.
MAMBA_AVAILABLE = False
try:
    _ = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "causal-conv1d>=1.4.0",
            "mamba-ssm>=2.2.2",
        ],
        check=True,
    )
    import mamba_ssm  # noqa: F401

    MAMBA_AVAILABLE = True
    print("mamba-ssm installed successfully.")
except Exception as exc:
    print("WARNING: mamba-ssm install failed; fallback paths will be used when available.")
    print("Reason:", exc)

def find_project_root() -> Path:
    search_roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]
    for base in search_roots:
        if not base.exists():
            continue
        for cfg_path in base.rglob("config.py"):
            root = cfg_path.parent
            if (root / "training" / "trainer.py").exists():
                return root
    raise FileNotFoundError(
        "Could not locate project root with config.py and training/trainer.py"
    )

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

if torch.cuda.is_available():
    n_gpu = torch.cuda.device_count()
    print(f"GPU count: {n_gpu}")
    for i in range(n_gpu):
        props = torch.cuda.get_device_properties(i)
        total_gb = props.total_memory / (1024 ** 3)
        with torch.cuda.device(i):
            free_b, total_b = torch.cuda.mem_get_info()
        free_gb = free_b / (1024 ** 3)
        print(f"GPU {i}: {props.name} | total={total_gb:.2f} GB | free={free_gb:.2f} GB")
else:
    print("No GPU detected.")


In [ ]:
print(f"ENVIRONMENT SETUP wall-clock: {time.time() - TIMER_ENV_START:.2f}s")


## 2. CONFIG


In [ ]:
import time

TIMER_CONFIG_START = time.time()

# Dataset
DATA_DIR = "/kaggle/input/godzilla-piano-tokenized"
MANIFEST_PATH = "/kaggle/input/godzilla-piano-tokenized/manifest.json"

# Which variant to train
VARIANT = "A"  # "A", "B", or "C"

# Model scale
D_MODEL = 512
N_LAYERS = 4

# Training
EPOCHS = 100
BATCH_SIZE = 16
LEARNING_RATE = 3e-4
WARMUP_STEPS = 500
GRAD_CLIP = 1.0
ACCUMULATION_STEPS = 2  # effective batch = 32
EARLY_STOPPING_PATIENCE = 10

# Dataset setup controls
MAX_MIDI_FILES = 15000  # reduce if hitting disk limits
METADATA_FILENAME = ""  # optional override if metadata filename differs
SKIP_DATASET_SETUP = False
RUN_SMOKE_TEST = True

# Generation
SEED_MIDI_PATH = "/kaggle/input/seed-midi/seed.mid"
GENERATION_TEMPERATURE = 0.9
GENERATION_TOP_P = 0.95
MAX_NEW_TOKENS = 512

# Output
OUTPUT_DIR = "/kaggle/working/outputs"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
SAVE_EVERY_N_EPOCHS = 10

# Stable geometry for this ablation setup
GLOBAL_SEED = 42
SEED_LENGTH = 256
CONTINUATION_LENGTH = 768

print("CONFIG loaded.")


In [ ]:
print(f"CONFIG wall-clock: {time.time() - TIMER_CONFIG_START:.2f}s")


## 3. IMPORTS AND MODEL SETUP


In [ ]:
TIMER_MODEL_START = time.time()
print("MODEL SETUP started...")


In [ ]:
import random
import numpy as np
import torch

from config import DataConfig, TrainConfig
from data.tokenizer_custom import CustomDeltaTokenizer
from training.trainer import Trainer

from model.variant_a import VariantAConfig, VariantAModel
from model.variant_b import VariantBConfig, VariantBModel
from model.variant_c import VariantCConfig, VariantCModel


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass




def _unwrap(model_or_wrapper):
    return model_or_wrapper.module if isinstance(model_or_wrapper, torch.nn.DataParallel) else model_or_wrapper

def build_model_for_variant(variant: str):
    v = variant.strip().upper()
    max_seq = SEED_LENGTH + CONTINUATION_LENGTH
    if v == "A":
        cfg = VariantAConfig(vocab_size=155, d_model=D_MODEL, n_layers=N_LAYERS, max_sequence_length=max_seq)
        return VariantAModel(cfg), cfg
    if v == "B":
        cfg = VariantBConfig(vocab_size=155, d_model=D_MODEL, n_layers=N_LAYERS, max_sequence_length=max_seq)
        return VariantBModel(cfg), cfg
    if v == "C":
        cfg = VariantCConfig(vocab_size=155, d_model=D_MODEL, n_layers=N_LAYERS, max_sequence_length=max_seq)
        return VariantCModel(cfg), cfg
    raise ValueError(f"Unsupported VARIANT={variant}")


set_global_seed(GLOBAL_SEED)

model, model_cfg = build_model_for_variant(VARIANT)
param_count = sum(p.numel() for p in model.parameters())
print(f"Variant {VARIANT.upper()} params: {param_count:,} ({param_count / 1e6:.2f}M)")

MANUAL_DATA_PARALLEL = False
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))
    MANUAL_DATA_PARALLEL = True
    print(f"Enabled DataParallel across {torch.cuda.device_count()} GPUs")
else:
    print("Single-device mode")

model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

if VARIANT.strip().upper() == "A":
    from model.blocks.gdn_block import GDN_AVAILABLE

    core_model = _unwrap(model)
    fallback_active = any(
        bool(getattr(mod, "using_fallback", False))
        for mod in core_model.modules()
        if hasattr(mod, "using_fallback")
    )
    if (not GDN_AVAILABLE) or fallback_active:
        print(
            "WARNING: Variant A GDN fallback is active because flash-linear-attention/GDN is unavailable. "
            "Training proceeds with fallback approximation blocks."
        )


In [ ]:
print(f"MODEL SETUP wall-clock: {time.time() - TIMER_MODEL_START:.2f}s")


## 3.5 TRIPLET MASKING SMOKE TEST


In [ ]:
TIMER_SMOKE_START = time.time()
if not RUN_SMOKE_TEST:
    print("RUN_SMOKE_TEST=False, skipping smoke test.")
else:
    print("Running triplet slot masking smoke test...")

    core_model = _unwrap(model)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    core_model.to(device)
    core_model.eval()

    t = CustomDeltaTokenizer(include_special_tokens=False)
    PAD = t.pad_id
    BOS = t.bos_id
    EOS = t.eos_id
    DELTA_LO, DELTA_HI = 0, 31
    PITCH_LO, PITCH_HI = 32, 119
    DUR_LO, DUR_HI = 120, 151
    print(
        f"Constants: PAD={PAD}, BOS={BOS}, EOS={EOS}, "
        f"delta=[{DELTA_LO},{DELTA_HI}], pitch=[{PITCH_LO},{PITCH_HI}], duration=[{DUR_LO},{DUR_HI}]"
    )

    dummy_tokens = []
    for i in range(30):
        dummy_tokens.extend(
            [
                DELTA_LO + (i % (DELTA_HI - DELTA_LO + 1)),
                PITCH_LO + (i % (PITCH_HI - PITCH_LO + 1)),
                DUR_LO + (i % (DUR_HI - DUR_LO + 1)),
            ]
        )

    token_tensor = torch.tensor(dummy_tokens, dtype=torch.long, device=device).unsqueeze(0)
    onset = torch.arange(token_tensor.shape[1], device=device, dtype=torch.float32).unsqueeze(0) * 0.1

    with torch.no_grad():
        logits = core_model(
            token_ids=token_tensor,
            onset_times=onset,
            memory=None,
            return_memory=False,
            position_offset=0,
        )

    failures = []
    for pos in range(token_tensor.shape[1]):
        slot = pos % 3
        masked = core_model._mask_logits_to_triplet_slot(logits[:, pos, :], slot)
        finite_ids = torch.isfinite(masked[0]).nonzero(as_tuple=False).view(-1).tolist()
        finite_set = set(int(x) for x in finite_ids)

        if slot == 0:
            expected = set(range(DELTA_LO, DELTA_HI + 1))
        elif slot == 1:
            expected = set(range(PITCH_LO, PITCH_HI + 1))
        else:
            expected = set(range(DUR_LO, DUR_HI + 1))

        if finite_set != expected:
            failures.append(
                {
                    "position": int(pos),
                    "slot": int(slot),
                    "expected_count": len(expected),
                    "actual_count": len(finite_set),
                }
            )

    if failures:
        print("SMOKE TEST FAIL")
        print("Failure count:", len(failures))
        print("First failure:", failures[0])
    else:
        print("SMOKE TEST PASS")

print(f"SMOKE TEST wall-clock: {time.time() - TIMER_SMOKE_START:.2f}s")


## 4. DATASET SETUP


In [ ]:
TIMER_DATASET_SETUP_START = time.time()
print("DATASET SETUP started...")


In [ ]:
import hashlib
import json
import random
import re
import subprocess
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
from huggingface_hub import hf_hub_download, list_repo_files
from tqdm.auto import tqdm

from data.tokenizer_custom import CustomDeltaTokenizer

# Step 1 — Install only what works.
_ = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "pretty_midi"],
    check=False,
)

REPO_ID = "projectlosangeles/Godzilla-MIDI-Dataset"
TOKENIZED_DIR = Path("/kaggle/working/tokenized")
MIDI_CACHE_DIR = Path("/kaggle/working/midi_cache")
METADATA_DIR = Path("/kaggle/working/metadata_cache")
MANIFEST_OUT = TOKENIZED_DIR / "manifest.json"

TOKENIZED_DIR.mkdir(parents=True, exist_ok=True)
MIDI_CACHE_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)


def disk_usage_gb() -> float:
    total = 0
    for d in (TOKENIZED_DIR, MIDI_CACHE_DIR, METADATA_DIR):
        if not d.exists():
            continue
        for p in d.rglob("*"):
            if p.is_file():
                total += p.stat().st_size
    return total / (1024 ** 3)


def looks_like_md5(value: str) -> bool:
    return bool(re.fullmatch(r"[0-9a-fA-F]{32}", value.strip()))


def maybe_int(v: Any) -> Optional[int]:
    if isinstance(v, bool):
        return None
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, float):
        if float(v).is_integer():
            return int(v)
        return None
    if isinstance(v, str):
        s = v.strip()
        if not s:
            return None
        if s.isdigit() or (s.startswith("-") and s[1:].isdigit()):
            return int(s)
        m = re.search(r"-?\d+", s)
        if m:
            return int(m.group(0))
    return None


def maybe_float(v: Any, default: float = 1.0) -> float:
    if isinstance(v, bool):
        return default
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v)
    if isinstance(v, str):
        s = v.strip()
        if not s:
            return default
        try:
            return float(s)
        except Exception:
            return default
    return default


def add_program(counts: Dict[int, float], program: Any, count: Any) -> None:
    p = maybe_int(program)
    if p is None or p < 0 or p > 127:
        return
    c = maybe_float(count, 0.0)
    if c <= 0.0:
        return
    counts[p] = counts.get(p, 0.0) + c


def ingest_payload(payload: Any, counts: Dict[int, float], prefer_second: bool = True) -> None:
    if isinstance(payload, dict):
        for k, v in payload.items():
            k_int = maybe_int(k)
            if k_int is not None:
                add_program(counts, k_int, v)
        for key in ("patch", "patch_id", "program", "program_id", "instrument", "instrument_id"):
            if key in payload:
                add_program(counts, payload.get(key), payload.get("count", payload.get("n", 1.0)))
    elif isinstance(payload, (list, tuple)):
        if not payload:
            return
        if len(payload) >= 3:
            a = maybe_int(payload[0])
            b = maybe_int(payload[1])
            c = maybe_float(payload[2], 1.0)
            chosen = b if (prefer_second and b is not None) else a
            if chosen is None:
                chosen = b
            if chosen is not None:
                add_program(counts, chosen, c)
            return
        if len(payload) == 2:
            add_program(counts, payload[0], payload[1])
        elif len(payload) == 1:
            add_program(counts, payload[0], 1.0)


def extract_program_counts(record: Dict[str, Any]) -> Dict[int, float]:
    counts: Dict[int, float] = {}
    direct_keys = [
        "pitches_patches_counts",
        "pitches-patches-counts",
        "patches_counts",
        "patch_counts",
        "program_counts",
        "instrument_counts",
        "patches",
        "programs",
        "instruments",
        "payload",
    ]
    for key in direct_keys:
        if key in record:
            ingest_payload(record[key], counts, prefer_second=("pitches" in key and "patch" in key))

    stack = [record]
    while stack:
        obj = stack.pop()
        if isinstance(obj, dict):
            for k, v in obj.items():
                low = str(k).lower()
                if any(tok in low for tok in ("patch", "program", "instrument")):
                    ingest_payload(v, counts, prefer_second=False)
                if isinstance(v, (dict, list, tuple)):
                    stack.append(v)
        elif isinstance(obj, (list, tuple)):
            for item in obj:
                if isinstance(item, (dict, list, tuple)):
                    stack.append(item)
    return {p: c for p, c in counts.items() if c > 0.0}


def extract_midi_path(record: Dict[str, Any]) -> Optional[str]:
    keys = [
        "path",
        "midi_path",
        "full_midi_path",
        "source_path",
        "file_path",
        "filepath",
        "file",
    ]
    for key in keys:
        v = record.get(key)
        if isinstance(v, str):
            low = v.lower()
            if low.endswith(".mid") or low.endswith(".midi"):
                return v
    for v in record.values():
        if isinstance(v, str):
            low = v.lower()
            if (low.endswith(".mid") or low.endswith(".midi")) and ("midi" in low):
                return v
    return None


def extract_md5(record: Dict[str, Any]) -> Optional[str]:
    for key in ("md5", "file_md5", "midi_md5", "hash"):
        v = record.get(key)
        if isinstance(v, str) and looks_like_md5(v):
            return v.lower()
    path_val = extract_midi_path(record)
    if isinstance(path_val, str):
        stem = Path(path_val).stem
        if looks_like_md5(stem):
            return stem.lower()
    return None


def iter_from_obj(obj: Any):
    if isinstance(obj, list):
        for item in obj:
            if isinstance(item, dict):
                yield item
    elif isinstance(obj, dict):
        # Record-like dict.
        if any(
            k in obj
            for k in (
                "md5",
                "path",
                "midi_path",
                "pitches_patches_counts",
                "pitches-patches-counts",
            )
        ):
            yield obj

        # md5-keyed dict format.
        for k, v in obj.items():
            if isinstance(k, str) and looks_like_md5(k):
                if isinstance(v, dict):
                    rec = dict(v)
                    rec.setdefault("md5", k.lower())
                    yield rec
                else:
                    yield {"md5": k.lower(), "payload": v}
            elif isinstance(v, dict) and any(
                kk in v
                for kk in (
                    "md5",
                    "path",
                    "midi_path",
                    "pitches_patches_counts",
                    "pitches-patches-counts",
                )
            ):
                yield v


def iter_metadata_records(path: Path):
    suffix = path.suffix.lower()
    if suffix == ".jsonl":
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except Exception:
                    continue
                yield from iter_from_obj(obj)
    else:
        try:
            obj = json.loads(path.read_text(encoding="utf-8"))
            yield from iter_from_obj(obj)
        except Exception:
            with path.open("r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        obj = json.loads(line)
                    except Exception:
                        continue
                    yield from iter_from_obj(obj)


def normalize_path(path: str) -> str:
    return str(path).replace("\\", "/").lstrip("./")


def midi_path_variants(midi_path: Optional[str], md5: Optional[str]) -> List[str]:
    out: List[str] = []

    if isinstance(midi_path, str) and midi_path.strip():
        base = normalize_path(midi_path)
        out.append(base)
        low = base.lower()
        if "midis/" in low:
            i = low.find("midis/")
            tail = base[i:]
            out.extend(
                [
                    tail,
                    tail.replace("midis/", "MIDIs/"),
                    f"Godzilla-MIDI-Dataset/{tail}",
                    f"Godzilla-MIDI-Dataset/{tail.replace('midis/', 'MIDIs/')}",
                ]
            )
        else:
            out.extend(
                [
                    f"MIDIs/{base}",
                    f"midis/{base}",
                    f"Godzilla-MIDI-Dataset/MIDIs/{base}",
                    f"Godzilla-MIDI-Dataset/midis/{base}",
                ]
            )

    if isinstance(md5, str) and md5:
        m = md5.lower()
        out.extend(
            [
                f"MIDIs/{m}.mid",
                f"MIDIs/{m}.midi",
                f"midis/{m}.mid",
                f"midis/{m}.midi",
                f"MIDIs/{m[:2]}/{m}.mid",
                f"MIDIs/{m[:2]}/{m}.midi",
                f"midis/{m[:2]}/{m}.mid",
                f"midis/{m[:2]}/{m}.midi",
            ]
        )

    dedup: List[str] = []
    seen = set()
    for p in out:
        if p not in seen:
            seen.add(p)
            dedup.append(p)
    return dedup


def download_midi(midi_path: Optional[str], md5: Optional[str]) -> Tuple[Optional[Path], Optional[str]]:
    for filename in midi_path_variants(midi_path, md5):
        try:
            local = hf_hub_download(
                repo_id=REPO_ID,
                filename=filename,
                repo_type="dataset",
                local_dir=str(MIDI_CACHE_DIR),
            )
            lp = Path(str(local))
            if lp.exists():
                return lp, filename
        except Exception:
            continue
    return None, None


if SKIP_DATASET_SETUP and MANIFEST_OUT.exists():
    print("SKIP_DATASET_SETUP=True and manifest exists. Skipping dataset setup.")
    DATA_DIR = str(TOKENIZED_DIR)
    MANIFEST_PATH = str(MANIFEST_OUT)
else:
    # Step 2 — Download pitches-patches metadata directly.
    filename_candidates = []
    requested = str(METADATA_FILENAME).strip()
    if requested:
        filename_candidates.append(requested)
    filename_candidates.extend(
        [
            "DATA/Pitches Patches Counts/pitches_patches_counts.jsonl",
            "DATA/Pitches Patches Counts/pitches_patches_counts.json",
            "DATA/Pitches_Patches_Counts/pitches_patches_counts.jsonl",
        ]
    )

    dedup = []
    seen = set()
    for fn in filename_candidates:
        if fn not in seen:
            seen.add(fn)
            dedup.append(fn)

    metadata_path: Optional[Path] = None
    metadata_filename_used = ""
    for filename in dedup:
        try:
            local = hf_hub_download(
                repo_id=REPO_ID,
                filename=filename,
                repo_type="dataset",
                local_dir=str(METADATA_DIR),
            )
            lp = Path(str(local))
            if lp.exists():
                metadata_path = lp
                metadata_filename_used = filename
                break
        except Exception:
            continue

    if metadata_path is None:
        files = list(list_repo_files(REPO_ID, repo_type="dataset"))
        pp_files = sorted(
            [
                f
                for f in files
                if ("pitches" in f.lower()) or ("patches" in f.lower())
            ]
        )
        print("Could not download metadata with default filenames.")
        print("Files containing 'pitches' or 'patches':")
        if not pp_files:
            print("  <none found>")
        else:
            for f in pp_files:
                print(" -", f)
        print(
            "ACTION REQUIRED: Update METADATA_FILENAME in CONFIG with the "
            "correct path shown above."
        )
        raise RuntimeError("Metadata filename unresolved.")

    print(f"Metadata downloaded: {metadata_filename_used}")
    print(f"Local metadata path: {metadata_path}")

    # Step 3 — Parse metadata and filter piano-only files.
    PIANO_PROGRAMS = set(range(8))
    piano_candidates: List[Dict[str, Any]] = []
    scanned = 0

    for rec in iter_metadata_records(metadata_path):
        scanned += 1
        if not isinstance(rec, dict):
            continue

        pcounts = extract_program_counts(rec)
        if not pcounts:
            continue

        programs = sorted(int(p) for p in pcounts.keys())
        if not programs:
            continue
        if not all(p in PIANO_PROGRAMS for p in programs):
            continue

        midi_path = extract_midi_path(rec)
        md5 = extract_md5(rec)
        if midi_path is None and md5 is None:
            continue

        piano_candidates.append(
            {
                "md5": md5,
                "midi_path": midi_path,
                "programs": programs,
            }
        )

    print(f"Scanned metadata rows: {scanned:,}")
    print(f"Piano-only candidates found: {len(piano_candidates):,}")

    random.seed(GLOBAL_SEED)
    random.shuffle(piano_candidates)
    selected = piano_candidates[: int(MAX_MIDI_FILES)]
    print(f"Selected candidates (cap={MAX_MIDI_FILES:,}): {len(selected):,}")

    # Step 4 — Download and tokenize MIDIs.
    tokenizer = CustomDeltaTokenizer(include_special_tokens=False)

    processed = 0
    skipped = 0
    failures = 0
    too_short = 0
    manifest_rows: List[Dict[str, Any]] = []

    pbar = tqdm(selected, desc="Download + tokenize", unit="file")
    for cand in pbar:
        if disk_usage_gb() > 14.0:
            print("WARNING: Disk usage exceeded 14GB. Stopping dataset setup early.")
            break

        processed += 1
        cand_md5 = cand.get("md5") if isinstance(cand.get("md5"), str) else None
        cand_path = (
            cand.get("midi_path") if isinstance(cand.get("midi_path"), str) else None
        )

        local_midi, repo_filename = download_midi(cand_path, cand_md5)
        if local_midi is None:
            skipped += 1
            failures += 1
            continue

        try:
            token_ids, onsets, durations = tokenizer.encode_with_time_features(local_midi)
        except Exception:
            skipped += 1
            failures += 1
            try:
                if str(local_midi).startswith(str(MIDI_CACHE_DIR)) and local_midi.exists():
                    local_midi.unlink()
            except Exception:
                pass
            continue

        if len(token_ids) < 64 * 3:
            skipped += 1
            too_short += 1
            try:
                if str(local_midi).startswith(str(MIDI_CACHE_DIR)) and local_midi.exists():
                    local_midi.unlink()
            except Exception:
                pass
            continue

        out_md5 = cand_md5
        if not isinstance(out_md5, str) or not looks_like_md5(out_md5):
            out_md5 = hashlib.md5(local_midi.read_bytes()).hexdigest()
        out_md5 = out_md5.lower()

        npz_path = TOKENIZED_DIR / f"{out_md5}.npz"
        np.savez_compressed(
            npz_path,
            tokens=np.asarray(token_ids, dtype=np.int16),
            onsets=np.asarray(onsets, dtype=np.float32),
            durations=np.asarray(durations, dtype=np.float32),
        )

        manifest_rows.append(
            {
                "md5": out_md5,
                "npz_path": npz_path.name,
                "length": int(len(token_ids)),
                "source_path": repo_filename if isinstance(repo_filename, str) else cand_path,
            }
        )

        try:
            if str(local_midi).startswith(str(MIDI_CACHE_DIR)) and local_midi.exists():
                local_midi.unlink()
        except Exception:
            pass

        if processed % 500 == 0:
            print(
                f"Processed={processed:,}, Success={len(manifest_rows):,}, "
                f"Skipped={skipped:,}, DiskUsed={disk_usage_gb():.2f} GB"
            )

    if not manifest_rows:
        raise RuntimeError("No tokenized files produced during dataset setup.")

    # Step 5 — Write manifest and rewire paths.
    MANIFEST_OUT.write_text(json.dumps(manifest_rows, indent=2), encoding="utf-8")
    DATA_DIR = str(TOKENIZED_DIR)
    MANIFEST_PATH = str(MANIFEST_OUT)

    print(f"Saved tokenized dir: {TOKENIZED_DIR}")
    print(f"Saved manifest: {MANIFEST_OUT}")
    print(
        f"Final stats -> processed={processed:,}, success={len(manifest_rows):,}, "
        f"skipped={skipped:,}, failures={failures:,}, too_short={too_short:,}, "
        f"disk={disk_usage_gb():.2f} GB"
    )
    print(f"Runtime DATA_DIR: {DATA_DIR}")
    print(f"Runtime MANIFEST_PATH: {MANIFEST_PATH}")


In [ ]:
print(f"DATASET SETUP wall-clock: {time.time() - TIMER_DATASET_SETUP_START:.2f}s")


## 5. DATA LOADING


In [ ]:
TIMER_DATA_START = time.time()
print("DATA LOADING started...")


In [ ]:
import json
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import torch
from torch.utils.data import DataLoader

from data.dataset import PianoDataset

manifest_path = Path(MANIFEST_PATH)
if not manifest_path.exists():
    raise FileNotFoundError(f"Manifest not found: {manifest_path}")

with manifest_path.open("r", encoding="utf-8") as f:
    raw_manifest = json.load(f)

if not isinstance(raw_manifest, list) or not raw_manifest:
    raise RuntimeError("Manifest is empty or invalid")

resolved_manifest: List[Dict[str, Any]] = []
for row in raw_manifest:
    if not isinstance(row, dict):
        continue

    npz_path = row.get("npz_path")
    if isinstance(npz_path, str) and npz_path:
        p = Path(npz_path)
        if not p.is_absolute():
            p = Path(DATA_DIR) / p
    else:
        md5 = str(row.get("md5", "")).strip()
        if not md5:
            continue
        p = Path(DATA_DIR) / "npz" / f"{md5}.npz"

    if not p.exists():
        continue

    length = int(row.get("length", row.get("tokens", -1)))
    if length <= 0:
        with np.load(p, allow_pickle=False) as pack:
            length = int(pack["tokens"].shape[0])

    resolved_manifest.append(
        {
            "piece_id": str(row.get("md5", p.stem)),
            "tokens_path": str(p.resolve()),
            "onset_times_path": "",
            "durations_path": "",
            "length": int(length),
            "tokens": int(length),
            "source": "godzilla_piano",
        }
    )

if not resolved_manifest:
    raise RuntimeError("No valid .npz files resolved from manifest")

lengths = np.asarray([int(m["length"]) for m in resolved_manifest], dtype=np.int64)
print(f"Total files: {len(resolved_manifest):,}")
print(f"Length mean/min/max: {lengths.mean():.2f}/{lengths.min()}/{lengths.max()}")
print("Length percentiles p50/p90/p99:", np.percentile(lengths, [50, 90, 99]).tolist())

idx = np.arange(len(resolved_manifest))
rng = np.random.default_rng(GLOBAL_SEED)
rng.shuffle(idx)

n_val = max(1, int(0.1 * len(idx)))
val_idx = set(idx[:n_val].tolist())
train_manifest = [resolved_manifest[i] for i in range(len(resolved_manifest)) if i not in val_idx]
val_manifest = [resolved_manifest[i] for i in range(len(resolved_manifest)) if i in val_idx]

print(f"Train files: {len(train_manifest):,}")
print(f"Val files: {len(val_manifest):,}")

tokenizer = CustomDeltaTokenizer(include_special_tokens=False)



class NpzWindowDataset(PianoDataset):
    # Adapt existing PianoDataset windowing to .npz tokenized files.

    def __getitem__(self, idx: int):
        item = self.manifest[idx]
        npz_path = Path(str(item["tokens_path"]))
        with np.load(npz_path, allow_pickle=False) as pack:
            token_seq = np.asarray(pack["tokens"], dtype=np.int64)
            onset_seq = np.asarray(pack["onsets"], dtype=np.float32)
            duration_seq = np.asarray(pack["durations"], dtype=np.float32)

        total_needed = self.data_config.seed_length + self.data_config.continuation_length
        if token_seq.shape[0] < total_needed:
            raise RuntimeError(f"Piece {npz_path} shorter than required window {total_needed}")

        max_start = int(token_seq.shape[0] - total_needed)
        raw_start = self.rng.randint(0, max_start) if max_start > 0 else 0
        start = self._snap_to_triplet_boundary(raw_start, max_start)
        if start % 3 != 0:
            raise AssertionError(f"Triplet boundary violation: {start}")

        seed = token_seq[start : start + self.data_config.seed_length]
        cont = token_seq[start + self.data_config.seed_length : start + total_needed]
        onset = onset_seq[start : start + total_needed]
        duration = duration_seq[start : start + total_needed]

        seed_t = torch.from_numpy(seed.astype(np.int64, copy=False))
        cont_t = torch.from_numpy(cont.astype(np.int64, copy=False))
        onset_t = torch.from_numpy(onset.astype(np.float32, copy=False))
        duration_t = torch.from_numpy(duration.astype(np.float32, copy=False))

        return {
            "seed": seed_t,
            "continuation": cont_t,
            "token_ids": torch.cat([seed_t, cont_t], dim=0),
            "onset_times": onset_t,
            "durations": duration_t,
            "new_piece": torch.tensor(True),
        }


data_config = DataConfig(
    tokenizer_path="",
    processed_path=str(Path(DATA_DIR).resolve()),
    vocab_size=155,
    tokenization_strategy="custom_delta",
    seed_length=SEED_LENGTH,
    continuation_length=CONTINUATION_LENGTH,
    max_sequence_length=SEED_LENGTH + CONTINUATION_LENGTH,
    use_continuous_time=True,
    time_feature_fallback_step_seconds=0.1,
)

train_dataset = NpzWindowDataset(train_manifest, data_config, seed=GLOBAL_SEED)
val_dataset = NpzWindowDataset(val_manifest, data_config, seed=GLOBAL_SEED + 1)

num_workers = 2
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(num_workers > 0),
    collate_fn=PianoDataset.collate_fn,
    drop_last=False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(num_workers > 0),
    collate_fn=PianoDataset.collate_fn,
    drop_last=False,
)

sample_batch = next(iter(train_loader))
print("Sample batch shape check:")
for k, v in sample_batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {tuple(v.shape)} | {v.dtype}")


In [ ]:
print("Running tokenizer roundtrip assertion...")

manual_tokens = []
for i in range(16):
    delta_tok = 0 if i == 0 else 17
    pitch_tok = 32 + [39, 43, 46, 51][i % 4]
    dur_tok = 126
    manual_tokens.extend([delta_tok, pitch_tok, dur_tok])

pm = tokenizer.decode(manual_tokens)
decoded_notes = []
for inst in pm.instruments:
    if inst.is_drum:
        continue
    decoded_notes.extend(inst.notes)
decoded_notes.sort(key=lambda n: (n.start, n.pitch))

rt_pass = True
rt_msgs = []
if len(decoded_notes) != 16:
    rt_pass = False
    rt_msgs.append(f"note_count mismatch: got {len(decoded_notes)} expected 16")

decoded_pitches = [int(n.pitch) for n in decoded_notes]
expected_pitch_set = {60, 64, 67, 72}
if decoded_pitches and not set(decoded_pitches).issubset(expected_pitch_set):
    rt_pass = False
    rt_msgs.append(
        f"pitch set mismatch: got={sorted(set(decoded_pitches))} expected subset={sorted(expected_pitch_set)}"
    )

if len(decoded_notes) >= 2:
    deltas = [decoded_notes[i].start - decoded_notes[i - 1].start for i in range(1, len(decoded_notes))]
    mean_delta = float(np.mean(deltas))
    if not (0.2 <= mean_delta <= 1.0):
        rt_pass = False
        rt_msgs.append(f"timing mismatch: mean delta {mean_delta:.3f} outside [0.2, 1.0]")

if rt_pass:
    print("Tokenizer roundtrip PASS")
else:
    print("Tokenizer roundtrip FAIL")
    for m in rt_msgs:
        print(" -", m)


In [ ]:
print(f"DATA LOADING wall-clock: {time.time() - TIMER_DATA_START:.2f}s")


## 6. TRAINING


In [ ]:
TIMER_TRAIN_START = time.time()
print("TRAINING started...")


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

train_config = TrainConfig(
    batch_size=BATCH_SIZE,
    grad_accumulation_steps=ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_grad_norm=GRAD_CLIP,
    max_epochs=EPOCHS,
    save_every_n_epochs=SAVE_EVERY_N_EPOCHS,
    checkpoint_dir=CHECKPOINT_DIR,
    use_wandb=False,
    seed=GLOBAL_SEED,
    device="cuda" if torch.cuda.is_available() else "cpu",
    val_generation_check=False,
)

# Avoid nested DataParallel if we already wrapped manually.
setattr(train_config, "_enable_data_parallel", not MANUAL_DATA_PARALLEL)
setattr(train_config, "_force_num_workers", 0)

print("Trainer live loss logs appear every 100 optimizer steps.")
if isinstance(model, torch.nn.DataParallel):
    print("Checkpoint path: DataParallel active -> saving/loading unwrapped module state_dict (model.module).")
else:
    print("Checkpoint path: single model -> saving/loading direct model state_dict.")

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_config,
    data_config=data_config,
    tokenizer=tokenizer,
)

# Early stopping wrapper around per-epoch training.
core_save_model = trainer._unwrap_model()
if isinstance(trainer.model, torch.nn.DataParallel):
    verified = core_save_model is trainer.model.module
    print(f"Checkpoint save verification: DataParallel path -> unwrap==module: {verified}")
else:
    print("Checkpoint save verification: single-model path.")

history = trainer.history
best_val = float('inf')
epochs_no_improve = 0
for epoch in range(1, EPOCHS + 1):
    trainer._run_one_epoch(epoch=epoch, max_epochs=EPOCHS)
    current_val = float(trainer.history['val_loss'][-1]) if trainer.history['val_loss'] else float('inf')
    if current_val + 1e-8 < best_val:
        best_val = current_val
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    if epochs_no_improve >= int(EARLY_STOPPING_PATIENCE):
        print(
            f"Early stopping triggered at epoch {epoch} (no val_loss improvement for {EARLY_STOPPING_PATIENCE} epoch(s))."
        )
        break
# Save final explicit checkpoint snapshot after early stop / completion.
final_val = float(trainer.history['val_loss'][-1]) if trainer.history['val_loss'] else float('inf')
trainer.save_checkpoint(epoch=int(getattr(trainer, 'current_epoch', EPOCHS)), val_loss=final_val, tag='final_earlystop')
print("Saved final checkpoint tag: final_earlystop")

history_path = Path(OUTPUT_DIR) / f"history_{VARIANT.upper()}.json"
with history_path.open("w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)
print("Saved history:", history_path.resolve())

curve_path = Path(OUTPUT_DIR) / f"loss_curve_{VARIANT.upper()}.png"
plt.figure(figsize=(10, 5))
plt.plot(history.get("train_loss", []), label="train_loss")
plt.plot(history.get("val_loss", []), label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"Train/Val Loss Variant {VARIANT.upper()}")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(curve_path, dpi=150)
plt.close()

print("Saved loss curve:", curve_path.resolve())


In [ ]:
print(f"TRAINING wall-clock: {time.time() - TIMER_TRAIN_START:.2f}s")


## 7. GENERATION


In [ ]:
TIMER_GEN_START = time.time()
print("GENERATION started...")


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pretty_midi

best_ckpt = Path(CHECKPOINT_DIR) / "best.safetensors"
latest_ckpt = Path(CHECKPOINT_DIR) / "latest.safetensors"

if best_ckpt.exists():
    ckpt_to_load = best_ckpt
elif latest_ckpt.exists():
    ckpt_to_load = latest_ckpt
else:
    raise FileNotFoundError("No best/latest checkpoint found")

_ = trainer.load_checkpoint(str(ckpt_to_load))
print("Loaded checkpoint:", ckpt_to_load.resolve())

# Correct DataParallel handling: unwrap before calling generation APIs.
core_model = _unwrap(trainer._unwrap_model())
if torch.cuda.is_available():
    core_model = core_model.to(torch.device('cuda:0'))
    print('Generation path: running unwrapped model on cuda:0 for stable CfC hidden-state threading.')
else:
    print('Generation path: running unwrapped model on CPU.')

seed_path = Path(SEED_MIDI_PATH)
if not seed_path.exists():
    raise FileNotFoundError(f"Seed MIDI not found: {seed_path}")

seed_ids, seed_onsets, _seed_durations = tokenizer.encode_with_time_features(seed_path)
if not seed_ids:
    raise RuntimeError("Seed MIDI tokenized to empty sequence")

seed_n = min(SEED_LENGTH, len(seed_ids))
seed_n = seed_n - (seed_n % 3)
if seed_n < 3:
    raise RuntimeError("Seed MIDI too short for triplet generation")

generated_ids = core_model.generate(
    seed_tokens=seed_ids[:seed_n],
    seed_onset_times=np.asarray(seed_onsets[:seed_n], dtype=np.float32),
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=GENERATION_TEMPERATURE,
    top_p=GENERATION_TOP_P,
    top_k=50,
    repetition_penalty=1.1,
    repetition_window=64,
    min_tokens_to_keep=3,
    step_seconds=0.1,
    token_id_to_events=tokenizer.decode_token_id_events,
)

out_midi = Path(OUTPUT_DIR) / f"generated_{VARIANT.upper()}.mid"
tokenizer.decode(generated_ids, out_midi)
print("Saved generated MIDI:", out_midi.resolve())

pm = pretty_midi.PrettyMIDI(str(out_midi))
roll = pm.get_piano_roll(fs=25)

plt.figure(figsize=(14, 5))
plt.imshow(np.log1p(roll), aspect="auto", origin="lower", cmap="magma")
plt.colorbar(label="log1p(velocity)")
plt.xlabel("Time frames")
plt.ylabel("MIDI pitch")
plt.title(f"Generated Piano Roll Variant {VARIANT.upper()}")
plt.tight_layout()

roll_path = Path(OUTPUT_DIR) / f"piano_roll_{VARIANT.upper()}.png"
plt.savefig(roll_path, dpi=150)
plt.close()
print("Saved piano roll:", roll_path.resolve())

def triplet_violations(tokens):
    bad = 0
    for i, tok in enumerate(tokens):
        slot = i % 3
        t = int(tok)
        if slot == 0 and not (0 <= t <= 31):
            bad += 1
        elif slot == 1 and not (32 <= t <= 119):
            bad += 1
        elif slot == 2 and not (120 <= t <= 151):
            bad += 1
    return bad

viol = triplet_violations(generated_ids)
print("Generated token sequence stats:")
print("  length:", len(generated_ids))
print("  unique tokens:", len(set(int(t) for t in generated_ids)))
print("  triplet integrity check passed:", viol == 0)
print("  triplet violations:", viol)


In [ ]:
print(f"GENERATION wall-clock: {time.time() - TIMER_GEN_START:.2f}s")


## 8. RESUME FROM CHECKPOINT (separate cell)

Run this cell independently to resume from `latest.safetensors` in `CHECKPOINT_DIR`.


In [ ]:
import time
from pathlib import Path

import json
import numpy as np
import torch
from torch.utils.data import DataLoader

from config import DataConfig, TrainConfig
from data.dataset import PianoDataset
from data.tokenizer_custom import CustomDeltaTokenizer
from training.trainer import Trainer

from model.variant_a import VariantAConfig, VariantAModel
from model.variant_b import VariantBConfig, VariantBModel
from model.variant_c import VariantCConfig, VariantCModel

TIMER_RESUME_START = time.time()
print("RESUME started...")

max_seq = SEED_LENGTH + CONTINUATION_LENGTH
v = VARIANT.strip().upper()
if v == "A":
    resume_core_model = VariantAModel(VariantAConfig(vocab_size=155, d_model=D_MODEL, n_layers=N_LAYERS, max_sequence_length=max_seq))
elif v == "B":
    resume_core_model = VariantBModel(VariantBConfig(vocab_size=155, d_model=D_MODEL, n_layers=N_LAYERS, max_sequence_length=max_seq))
elif v == "C":
    resume_core_model = VariantCModel(VariantCConfig(vocab_size=155, d_model=D_MODEL, n_layers=N_LAYERS, max_sequence_length=max_seq))
else:
    raise ValueError(f"Unsupported VARIANT={VARIANT}")

resume_core_model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

manifest_path = Path(MANIFEST_PATH)
with manifest_path.open("r", encoding="utf-8") as f:
    raw_manifest = json.load(f)

resolved_manifest = []
for row in raw_manifest:
    if not isinstance(row, dict):
        continue
    npz_path = row.get("npz_path")
    if isinstance(npz_path, str) and npz_path:
        p = Path(npz_path)
        if not p.is_absolute():
            p = Path(DATA_DIR) / p
    else:
        md5 = str(row.get("md5", "")).strip()
        if not md5:
            continue
        p = Path(DATA_DIR) / "npz" / f"{md5}.npz"
    if not p.exists():
        continue
    length = int(row.get("length", row.get("tokens", -1)))
    if length <= 0:
        with np.load(p, allow_pickle=False) as pack:
            length = int(pack["tokens"].shape[0])
    resolved_manifest.append(
        {
            "piece_id": str(row.get("md5", p.stem)),
            "tokens_path": str(p.resolve()),
            "onset_times_path": "",
            "durations_path": "",
            "length": int(length),
            "tokens": int(length),
            "source": "godzilla_piano",
        }
    )

idx = np.arange(len(resolved_manifest))
rng = np.random.default_rng(GLOBAL_SEED)
rng.shuffle(idx)
n_val = max(1, int(0.1 * len(idx)))
val_idx = set(idx[:n_val].tolist())
train_manifest = [resolved_manifest[i] for i in range(len(resolved_manifest)) if i not in val_idx]
val_manifest = [resolved_manifest[i] for i in range(len(resolved_manifest)) if i in val_idx]

class ResumeNpzDataset(PianoDataset):
    def __getitem__(self, idx: int):
        item = self.manifest[idx]
        npz_path = Path(str(item["tokens_path"]))
        with np.load(npz_path, allow_pickle=False) as pack:
            token_seq = np.asarray(pack["tokens"], dtype=np.int64)
            onset_seq = np.asarray(pack["onsets"], dtype=np.float32)
            duration_seq = np.asarray(pack["durations"], dtype=np.float32)

        total_needed = self.data_config.seed_length + self.data_config.continuation_length
        max_start = int(token_seq.shape[0] - total_needed)
        raw_start = self.rng.randint(0, max_start) if max_start > 0 else 0
        start = self._snap_to_triplet_boundary(raw_start, max_start)

        seed = token_seq[start : start + self.data_config.seed_length]
        cont = token_seq[start + self.data_config.seed_length : start + total_needed]
        onset = onset_seq[start : start + total_needed]
        duration = duration_seq[start : start + total_needed]

        seed_t = torch.from_numpy(seed.astype(np.int64, copy=False))
        cont_t = torch.from_numpy(cont.astype(np.int64, copy=False))
        onset_t = torch.from_numpy(onset.astype(np.float32, copy=False))
        duration_t = torch.from_numpy(duration.astype(np.float32, copy=False))

        return {
            "seed": seed_t,
            "continuation": cont_t,
            "token_ids": torch.cat([seed_t, cont_t], dim=0),
            "onset_times": onset_t,
            "durations": duration_t,
            "new_piece": torch.tensor(True),
        }

resume_data_config = DataConfig(
    tokenizer_path="",
    processed_path=str(Path(DATA_DIR).resolve()),
    vocab_size=155,
    tokenization_strategy="custom_delta",
    seed_length=SEED_LENGTH,
    continuation_length=CONTINUATION_LENGTH,
    max_sequence_length=max_seq,
    use_continuous_time=True,
    time_feature_fallback_step_seconds=0.1,
)

train_dataset = ResumeNpzDataset(train_manifest, resume_data_config, seed=GLOBAL_SEED)
val_dataset = ResumeNpzDataset(val_manifest, resume_data_config, seed=GLOBAL_SEED + 1)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
    collate_fn=PianoDataset.collate_fn,
    drop_last=False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
    collate_fn=PianoDataset.collate_fn,
    drop_last=False,
)

resume_train_config = TrainConfig(
    batch_size=BATCH_SIZE,
    grad_accumulation_steps=ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_grad_norm=GRAD_CLIP,
    max_epochs=EPOCHS,
    save_every_n_epochs=SAVE_EVERY_N_EPOCHS,
    checkpoint_dir=CHECKPOINT_DIR,
    use_wandb=False,
    seed=GLOBAL_SEED,
    device="cuda" if torch.cuda.is_available() else "cpu",
    val_generation_check=False,
)

# Keep unwrapped for loading.
setattr(resume_train_config, "_enable_data_parallel", False)
setattr(resume_train_config, "_force_num_workers", 0)

resume_trainer = Trainer(
    model=resume_core_model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=resume_train_config,
    data_config=resume_data_config,
    tokenizer=CustomDeltaTokenizer(include_special_tokens=False),
)

latest_ckpt = Path(CHECKPOINT_DIR) / "latest.safetensors"
if not latest_ckpt.exists():
    raise FileNotFoundError(f"Latest checkpoint not found: {latest_ckpt}")

state = resume_trainer.load_checkpoint(str(latest_ckpt))
print("Resume checkpoint path: loaded into unwrapped model before any DataParallel wrapping.")

if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    core_loaded = resume_trainer._unwrap_model()
    resume_trainer.model = torch.nn.DataParallel(
        core_loaded,
        device_ids=list(range(torch.cuda.device_count())),
    )
    resume_trainer.model.to(torch.device("cuda"))
    print("Resume checkpoint path: re-wrapped model after load.")
else:
    print("Resume checkpoint path: single-GPU/CPU, no wrap applied.")

start_epoch = int(state.get("epoch", 0))
print(f"Loaded latest checkpoint epoch: {start_epoch}")

remaining = max(0, EPOCHS - start_epoch)
if remaining == 0:
    print("No remaining epochs to run.")
else:
    print(f"Resuming for {remaining} epoch(s)...")
    _ = resume_trainer.train_n_epochs(remaining, start_epoch=start_epoch)
    print("Resume training complete.")

print(f"RESUME wall-clock: {time.time() - TIMER_RESUME_START:.2f}s")
